# Лекција 12 - Смањење историје четa помоћу агентске радне табле

Овај бележник демонстрира како управљати контекстом у дугим разговорима коришћењем Microsoft Agent Framework-а. Како разговори расту, број токена се повећава — на крају премашујући прозор контекста модела. Ово решавамо помоћу **шаблона за сумирање контекста** и **агентске радне табле** за упорну меморију.

## Шта ћете научити:
1. **Зашто је важно управљање контекстом**: Разумевање ограничења токена и прозора контекста
2. **Агенти свесни контекста**: Прављење агената који управљају својим контекстом разговора
3. **Шаблон за сумирање контекста**: Коришћење алата за кондензовање историје разговора
4. **Агентска радна табла**: Упорна меморија која опстаје упркос смањењу контекста

## Предуслови:
- Подешавање Azure OpenAI са конфигурисаним променљивим окружења
- Разумевање основних појмова о агентима из претходних лекција


## Подешавање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Зашто је управљање контекстом важно

Сваки LLM има ограничен **прозор контекста** — максималан број токена које може обрадити у једном захтеву. Како разговор са више кругова напредује:

- **Број токена расте линеарно** са сваким корисничким поруком и одговором асистента.
- **Токени у упиту доминирају трошковима** јер се цела хронологија поново шаље сваки пут.
- На крају разговор **прелази прозор контекста** и модел или скраћује или јавља грешку.

### Стратегије за управљање контекстом

| Стратегија | Како функционише | Компромис |
|---|---|---|
| **Скраћивање** | Избацују се најстарије поруке | Губи се ранији контекст |
| **Сажимање** | Ставе се старије поруке у сажетак | Неки детаљи се губе, али се кључне тачке чувају |
| **Скетчпад / Спољна меморија** | Кључне чињенице се чувају ван разговора | Захтева позиве алата, али опстаје при сваком смањењу |

У овом бележнику комбинујемо **сажимање** са **скетчпад алатом** како би агент могао да одржи континуитет чак и када се историја разговора сабије.


## Креирање агента свесног контекста


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Симулација дугог разговора

Хајде да прођемо кроз разговор са више корака како бисмо видели како се контекст акумулира. Агенат би требало да запамти кључне детаље (преференције, буџет, датуме путовања) током разговора и покаже континуитет.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Приметите како агент задржава контекст из претходних кругова — зна о Јапану, сушију, храмовима, фотографији, буџету од 3000 долара, соло путовању и априлском временском периоду. У кратком разговору ово добро функционише, али како разговор расте, слање целе историје постаје скупо.

Хајде да наставимо разговор са више кругова да видимо акумулацију контекста:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Образац за резимирање контекста

Како разговор расте, можемо користити **алат за резимирање** да сажмемо нагомилани контекст у компактни формат. Агенат позива овај алат да забележи кључне преференције тако да, чак и ако старије поруке буду уклоњене, суштинске информације буду сачуване.

Овај образац је основни елемент за сложеније смањење историје:
1. Агенат идентификује кључне чињенице из разговора
2. Позива алат за резимирање да их сачува
3. Старије поруке се могу безбедно уклонити јер резиме садржи оно што је важно

Испод дефинишемо алат `summarize_preferences` који агенат може позвати да забележи компактни резиме онога што је научио.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Резиме

У овој лекцији сте научили како да управљате контекстом у дуготрајним разговорима са агентом коришћењем Microsoft Agent Framework:

### Кључни појмови
- **Прозори контекста су ограничени** — сваки токен у историји разговора кошта и улази у лимит.
- **Алати за сумиратију** омогућавају агенту да сабере акумулирани контекст у компактне резимеје, смањујући коришћење токена док се чувају важне информације.
- **Белешке агента** обезбеђују трајну спољашњу меморију која опстане без обзира на смањење разговора.

### Шта сте направили
- **Агент свестан контекста** који одржава континуитет кроз вишеслојне разговоре
- **Алат за сумирање** (`summarize_preferences`) који бележи кључне детаље корисника у компактном формату
- **Вишеслојни разговор** који демонстрира одржавање контекста и руковање променама

### Примене у стварном свету
- **Ботови за корисничку подршку**: памте преференције током дугих сесија подршке
- **Персонални асистенти**: прате текуће пројекте без потребе да се поново објашњава контекст
- **Образовни тутори**: одржавају напредак ученика кроз бројне интеракције

### Следећи кораци
- Имплементирати пун алат за белешке са трајним чувањем у фајловима
- Додати аутоматско скраћивање историје након сумирања
- Комбиновати са векторским базама података за семантичко претраживање меморије
- Правити агенте који могу наставити разговоре дане касније са пуним контекстом


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
